# One wall in IFC
**COMPAS foundations · AEC Formats: IFC**

**Why this is needed:** a box describes a shape. IFC can describe an **IfcWall**, its building and floor, its units and its shape. Those meanings help different AEC applications exchange a model.

We create one real IFC4 wall with IfcOpenShell, save it and read it back. The COMPAS scene previews the same dimensions; it is constructed separately from those parameters, not extracted from the IFC geometry. We do not yet mesh an IFC model or analyse the wall.

**You already know:** primitives, JSON and the purpose of a model. This is a later foundation lesson; start with the box tutorial first. Dimensions are in metres.


In [ ]:
%pip install -q compas==2.15.1 pythreejs==2.4.2
%pip install -q compas_notebook==0.11.0 --no-deps


In [ ]:
try:
    from google.colab import output
except ImportError:
    pass
else:
    output.enable_custom_widget_manager()


## 1. Install the IFC reader and writer
BRG's browser runtime uses Python 3.13 / Pyodide 0.29.3. Its official IfcOpenShell WebAssembly wheel is different from a desktop Python wheel. The conditional below selects the matching package. The first run downloads it.


In [ ]:
import sys

if sys.platform == "emscripten":
    import micropip
    await micropip.install(
        "https://ifcopenshell.github.io/wasm-wheels/"
        "ifcopenshell-0.8.4-cp313-cp313-pyodide_2025_0_wasm32.whl"
    )
else:
    %pip install -q ifcopenshell==0.8.4

import ifcopenshell
from ifcopenshell.api import run
from importlib.metadata import version
print("IfcOpenShell package", version("ifcopenshell"))


## 2. Give the wall a place in the building
These entities form the hierarchy **Project → Site → Building → Storey**. Aggregation means one object is part of another. A building hierarchy is information that a standalone box does not contain.


In [ ]:
ifc = run("project.create_file", version="IFC4")
project = run("root.create_entity", ifc, ifc_class="IfcProject", name="First IFC")
site = run("root.create_entity", ifc, ifc_class="IfcSite", name="Site")
building = run("root.create_entity", ifc, ifc_class="IfcBuilding", name="Building")
storey = run("root.create_entity", ifc, ifc_class="IfcBuildingStorey", name="Ground floor")

for parent, child in [(project, site), (site, building), (building, storey)]:
    run("aggregate.assign_object", ifc, products=[child], relating_object=parent)


## 3. State the units and the shape context
We explicitly choose metres, square metres and cubic metres. `Body` identifies the physical 3D shape representation. This prevents another application from having to guess the meaning of the numbers.


In [ ]:
units = [run("unit.add_si_unit", ifc, unit_type=kind)
         for kind in ["LENGTHUNIT", "AREAUNIT", "VOLUMEUNIT"]]
run("unit.assign_unit", ifc, units=units)
context = run("context.add_context", ifc, context_type="Model")
body = run("context.add_context", ifc, context_type="Model",
           context_identifier="Body", target_view="MODEL_VIEW", parent=context)


## 4. Create the wall
The length, thickness and height drive both the IFC representation and the COMPAS preview. The wall has an identity, a placement and a floor container. No force or stiffness calculation is involved.


In [ ]:
length, thickness, height = 4.0, 0.2, 3.0  # metres
wall = run("root.create_entity", ifc, ifc_class="IfcWall", name="Wall A")
shape = run("geometry.add_wall_representation", ifc, context=body,
            length=length, height=height, thickness=thickness)
run("geometry.assign_representation", ifc, product=wall, representation=shape)
run("geometry.edit_object_placement", ifc, product=wall)
run("spatial.assign_container", ifc, products=[wall], relating_structure=storey)
print(wall.is_a(), wall.Name, wall.GlobalId)


## 5. Preview those dimensions with COMPAS
A centred COMPAS box is shifted so its minimum corner is `(0, 0, 0)`, like our IFC wall. This is a parameter-based illustration of the exported wall.


In [ ]:
from compas.geometry import Box, Frame
from compas_notebook import Viewer
from compas_notebook.config import Config

frame = Frame([length / 2, thickness / 2, height / 2], [1, 0, 0], [0, 1, 0])
preview = Box(xsize=length, ysize=thickness, zsize=height, frame=frame)
config = Config()
config.view.width = 700
config.view.height = 360
config.view.camera.position = [7, -8, 5]
config.view.camera.target = [length / 2, 0, height / 2]
viewer = Viewer(config=config)
viewer.scene.add(preview.to_mesh(), color="#5552bb", show_edges=True)
viewer.show()


## 6. Export and inspect
This writes an actual `.ifc` file. Download it from Jupyter's file browser. Reading it back should preserve the wall's IFC identity, representation and floor relationship. These checks do not claim full certification of a building model.


In [ ]:
ifc.write("first-wall.ifc")
loaded = ifcopenshell.open("first-wall.ifc")
loaded_walls = loaded.by_type("IfcWall")
assert len(loaded_walls) == 1
restored_wall = loaded_walls[0]
assert restored_wall.GlobalId == wall.GlobalId
assert restored_wall.Representation is not None
assert restored_wall.ContainedInStructure[0].RelatingStructure.Name == "Ground floor"
length_unit = next(unit for unit in loaded.by_type("IfcSIUnit")
                   if unit.UnitType == "LENGTHUNIT")
assert length_unit.Name == "METRE" and length_unit.Prefix is None
print(loaded.schema, restored_wall.Name)
print("Floor:", restored_wall.ContainedInStructure[0].RelatingStructure.Name)
print("Saved first-wall.ifc; length unit is metre.")


## Try one change
1. Change `height` to 2 m. Rerun the project, units, wall, preview and export cells in order (or use Run All Cells).
2. Change the wall's name. Read it back and inspect `restored_wall.Name`.
3. Explain which information would be lost if you exported only the box's vertices.

**References:** [IfcOpenShell official browser installation](https://docs.ifcopenshell.org/ifcopenshell-python/installation.html#web-assembly), [official WASM wheels](https://github.com/IfcOpenShell/wasm-wheels), [IfcOpenShell wall API](https://docs.ifcopenshell.org/autoapi/ifcopenshell/api/geometry/add_wall_representation/index.html), [buildingSMART IfcWall entity](https://ifc43-docs.standards.buildingsmart.org/IFC/RELEASE/IFC4x3/HTML/lexical/IfcWall.htm). Original BRG tutorial using IfcOpenShell's public API; export schema is IFC4.
